# Classical Reasoning System: entailment over rules, logic stories and causal graphs

**Task.** Given a knowledge base (context) and a query (statement), decide whether the query is entailed: `True` / `False`.

**Pipeline.** text → router → parser → representation → **classical reasoning** → True/False

| Dataset | Representation | Reasoning |
|---|---|---|
| RuleTaker (`jise/ruletaker`) | propositional literals + Horn rules | stratified forward chaining with negation-as-failure |
| FOLIO (`yale-nlp/FOLIO`) | FOL annotation → CNF clauses | resolution refutation (set of support, best-first, unification) |
| CounterBench (`CounterBench/CounterBench`) | boolean structural equations | abduction → action (do) → prediction |

No generative model is used; the decision always comes from the reasoning engine.

**To generate `submission.csv`:** set `TEST_PATH` in the last section and run it. The data-loading and experiment sections are optional for that step.

## 1. Public data (development and experiments only)
FOLIO is gated on Hugging Face: accept its terms on the dataset page and log in when the prompt appears.
If the datasets cannot be downloaded the notebook still works: experiment cells are skipped and the engine and submission cells run normally.

In [1]:
RUN_EXPERIMENTS = True
try:
    from datasets import load_dataset
    try:                                    # needed for the gated yale-nlp/FOLIO dataset
        from huggingface_hub import notebook_login
        notebook_login()
    except Exception:
        pass
    ds_rule_taker = load_dataset("jise/ruletaker")            # train / test / ood_test
    ds_folio = load_dataset("yale-nlp/FOLIO")                 # train / validation
    # CounterBench: load the question files one by one (the repo also holds
    # meta_model_*.json files with a different schema, which break a plain load_dataset)
    ds_counter_bench = {}
    for fn in ["data_balanced_alpha_V1.json", "data_balanced_backdoor_V2.json"]:
        try:
            ds_counter_bench[fn] = load_dataset("CounterBench/CounterBench", data_files=fn)["train"]
        except Exception as e:
            print(f"CounterBench file {fn} could not be loaded:", e)
    print("RuleTaker   :", {s: len(ds_rule_taker[s]) for s in ds_rule_taker})
    print("FOLIO       :", {s: len(ds_folio[s]) for s in ds_folio})
    print("CounterBench:", {f: len(d) for f, d in ds_counter_bench.items()})
except Exception as e:
    print("Public datasets not available, experiments will be skipped:", e)
    RUN_EXPERIMENTS = False

Generating train split: 0 examples [00:00, ? examples/s]

RuleTaker   : {'train': 1000, 'test': 250, 'ood_test': 375}
FOLIO       : {'train': 1001, 'validation': 203}
CounterBench: {'data_balanced_alpha_V1.json': 1000, 'data_balanced_backdoor_V2.json': 200}


## 2. Configuration

In [2]:
import os, re, sys, json, time, csv, random, itertools
from collections import defaultdict, Counter

SEED = 42
random.seed(SEED)

PROVER_TIME_LIMIT = 2.0      # seconds per FOL query (FOLIO resolution budget)
PROVER_MAX_CLAUSES = 4000

## 3. RuleTaker (generic templates): semantic parser + stratified forward chaining

In [3]:
VARWORDS = {"someone", "something", "they", "it", "them", "he", "she",
            "people", "things", "ones", "one", "everyone", "everything"}
GENERIC_NOUNS = {"things", "people", "ones", "someone", "something", "everything", "everyone"}
VERBSET = {"is", "are", "was", "were", "does", "do", "did", "can", "cannot", "has", "have"}
ARTICLES = {"a", "an"}

def lemma(w):
    if w in ("has", "have"): return "have"
    if w in ("is", "are", "was", "were", "be"): return "be"
    if w in ("does", "do"): return "do"
    for suf in ("ches", "shes", "sses", "xes", "oes", "zzes"):
        if w.endswith(suf): return w[:-2]
    if w.endswith("ies") and len(w) > 4: return w[:-3] + "y"
    if w.endswith("s") and not w.endswith("ss"): return w[:-1]
    return w

def looks_verb(w):
    return w in VERBSET or (w.endswith("s") and not w.endswith(("ss", "us")) and len(w) > 2)

def singular(w, nouns):
    if w in nouns: return w
    for cut in (1, 2):
        if w[:-cut] in nouns: return w[:-cut]
    if w.endswith("ies"): return w[:-3] + "y"
    if w.endswith(("ches", "shes", "sses", "xes")): return w[:-2]
    if w.endswith("s") and not w.endswith(("ss", "us")): return w[:-1]
    return w

def norm_sentence(s):
    s = s.strip().strip(".").strip()
    s = re.sub(r"\s+", " ", s)
    s = s.replace("n't", " not").replace("can not", "cannot")
    return s.lower()

def split_sentences(text):
    parts = re.split(r"(?<=[.!?])\s+|\n+", text.strip())
    return [p for p in (x.strip() for x in parts) if p]

class NLParser:
    """Rule-based semantic parser from template English into literals/rules."""
    def __init__(self, nouns=()):
        self.nouns = set(nouns)

    def term(self, words):
        if not words: return None
        t = " ".join(words)
        if t in VARWORDS: return "?x"
        return t

    def parse_subject(self, w):
        """Return (subject, rest) or (None, w) if no explicit subject."""
        if not w: return None, w
        if w[0] in VARWORDS: return "?x", w[1:]
        if w[0] == "the":
            for j in range(2, len(w)):
                if looks_verb(w[j]): return " ".join(w[:j]), w[j:]
            return None, w
        if len(w) > 1 and w[0] not in VERBSET and w[0] not in ("not", "all", "every", "each") and w[1] not in GENERIC_NOUNS:
            if w[1] in VERBSET and w[1] not in ("are", "were", "do", "have"):
                return w[0], w[1:]
            if w[1] not in VERBSET and looks_verb(w[1]) and not (len(w) > 2 and w[2] in VERBSET):
                return w[0], w[1:]
        return None, w

    def parse_vp(self, rest, subj, prev_verb="be", verb_first=False):
        """Parse a verb phrase into a literal."""
        neg = False
        if not rest: return None
        r = list(rest)
        if r[0] in ("is", "are", "was", "were"):
            verb = "be"; r = r[1:]
            if r and r[0] == "not": neg = True; r = r[1:]
        elif r[0] in ("does", "do", "did"):
            if len(r) > 1 and r[1] == "not": neg = True; r = r[2:]
            else: r = r[1:]
            if not r: return None
            verb = lemma(r[0]); r = r[1:]
        elif r[0] == "cannot":
            neg = True; verb = "can"; r = r[1:]
        elif r[0] == "can":
            verb = "can"; r = r[1:]
            if r and r[0] == "not": neg = True; r = r[1:]
        elif r[0] == "not":
            neg = True; r = r[1:]
            if r and looks_verb(r[0]) and r[0] not in VERBSET:
                verb = lemma(r[0]); r = r[1:]
            else:
                verb = prev_verb
        elif verb_first or looks_verb(r[0]):
            verb = lemma(r[0]); r = r[1:]
        else:
            verb = prev_verb
        if verb in ("be", "can"):
            if r and r[0] in ARTICLES: r = r[1:]
            if not r: return None
            if r[0] == "the" or (len(r) == 1 and r[0] in VARWORDS and r[0] not in ("one",)):
                return (neg, verb + ":=", subj, self.term(r)), verb
            comp = " ".join(singular(x, self.nouns) if (len(r) == 1 and verb == "be" and rest and rest[0] == "are") else x for x in r)
            return (neg, verb + ":" + comp, subj, None), verb
        if not r: return (neg, verb, subj, None), verb
        return (neg, verb, subj, self.term(r)), verb

    def parse_clause(self, text, prev_subj=None, prev_verb="be"):
        w = text.replace(",", " ").split()
        subj, rest = self.parse_subject(w)
        if subj is None:
            subj = prev_subj
            if subj is None: return None
        out = self.parse_vp(rest, subj, prev_verb)
        return out

    def parse_conj(self, text, prev_subj=None):
        lits, subj, verb = [], prev_subj, "be"
        for part in re.split(r"\band\b", text):
            part = part.strip(" ,")
            if not part: continue
            res = self.parse_clause(part, subj, verb)
            if res is None: return None
            lit, verb = res
            subj = lit[2]
            lits.append(lit)
        return lits

    def parse_generic_subject(self, words):
        words = [x for x in words if x not in ("all", "and", "every", "each")]
        if not words: return None
        conds = []
        head = words[-1]
        adjs = words[:-1]
        if head not in GENERIC_NOUNS:
            conds.append((False, "be:" + singular(head, self.nouns), "?x", None))
        i = 0
        while i < len(adjs):
            neg = False
            if adjs[i] == "not": neg = True; i += 1
            if i < len(adjs): conds.append((neg, "be:" + adjs[i], "?x", None))
            i += 1
        return conds

    def parse_sentence(self, s):
        """Return ('fact', lit) | ('rule', conds, concls) | None."""
        s = norm_sentence(s)
        m = re.match(r"^if (.+?),? then (.+)$", s)
        if m:
            conds = self.parse_conj(m.group(1))
            if not conds: return None
            concls = self.parse_conj(m.group(2), conds[0][2])
            if not concls: return None
            return ("rule", conds, concls)
        w = s.replace(",", " ").split()
        subj, rest = self.parse_subject(w)
        if subj is not None and subj != "?x":
            lits = self.parse_conj(" ".join(rest), subj)
            if lits and len(lits) == 1: return ("fact", lits[0])
            if lits: return ("facts", lits)
            return None
        # generic: "All big, red things are kind." / "Birds can fly." / "Kind people chase the dog."
        for j in range(1, len(w)):
            if w[j] in VERBSET or w[j - 1] in GENERIC_NOUNS:
                conds = self.parse_generic_subject(w[:j])
                if not conds: return None
                res = self.parse_conj(" ".join(w[j:]), "?x") if w[j] in VERBSET else None
                if res is None:
                    r = self.parse_vp(w[j:], "?x", "be", verb_first=True)
                    if r is None: return None
                    res = [r[0]]
                return ("rule", conds, res)
        return None

def collect_nouns(text):
    return set(re.findall(r"\b(?:a|an) ([a-z]+)\b", text.lower()))

def build_kb(context):
    parser = NLParser(collect_nouns(context))
    facts, rules, unparsed = [], [], []
    for s in split_sentences(context):
        p = parser.parse_sentence(s)
        if p is None: unparsed.append(s); continue
        if p[0] == "fact": facts.append(p[1])
        elif p[0] == "facts": facts.extend(p[1])
        else:
            for c in p[2]: rules.append((p[1], c))
    return facts, rules, unparsed, parser

# ---------------- Forward chaining engine ----------------
def is_var(t): return isinstance(t, str) and t.startswith("?")

def match(lit, fact, theta):
    """Unify a (possibly non-ground) literal with a ground fact."""
    if lit[0] != fact[0] or lit[1] != fact[1]: return None
    th = dict(theta)
    for a, b in ((lit[2], fact[2]), (lit[3], fact[3])):
        if is_var(a):
            if a in th:
                if th[a] != b: return None
            else: th[a] = b
        elif a != b: return None
    return th

def subst(lit, th):
    return (lit[0], lit[1], th.get(lit[2], lit[2]), th.get(lit[3], lit[3]))

class ForwardChainer:
    """Stratified forward chaining.
    mode='cwa' : negation-as-failure (closed world), negative conditions succeed
                 when the positive atom is NOT derivable at a lower stratum.
    mode='owa' : negation is classical; ~p must be explicitly derived."""
    def __init__(self, facts, rules, mode="cwa", max_iter=200):
        self.mode, self.rules, self.max_iter = mode, rules, max_iter
        self.trace = {}
        self.entities = set()
        for f in facts:
            self.entities.add(f[2])
            if f[3] is not None and not is_var(f[3]): self.entities.add(f[3])
        for conds, c in rules:
            for l in conds + [c]:
                for t in (l[2], l[3]):
                    if t is not None and not is_var(t): self.entities.add(t)
        if mode == "cwa":
            self.facts = set(f for f in facts if not f[0])
        else:
            self.facts = set(facts)
        for f in self.facts: self.trace[f] = ("given",)
        self.index = defaultdict(set)
        for f in self.facts: self.index[(f[0], f[1])].add(f)
        self.steps = 0
        self.run()

    def holds_naf(self, lit):
        pos = (False, lit[1], lit[2], lit[3])
        return pos not in self.facts

    def strata(self):
        preds = set()
        for conds, c in self.rules:
            preds.add(c[1]); preds.update(l[1] for l in conds)
        st = {p: 0 for p in preds}
        for _ in range(len(preds) + 1):
            changed = False
            for conds, c in self.rules:
                for l in conds:
                    need = st[l[1]] + (1 if (l[0] and self.mode == "cwa") else 0)
                    if st[c[1]] < need: st[c[1]] = need; changed = True
            if not changed: break
        return st

    def fire(self, conds, concl):
        pos = [l for l in conds if not (l[0] and self.mode == "cwa")]
        naf = [l for l in conds if (l[0] and self.mode == "cwa")]
        thetas = [{}]
        for l in pos:
            new = []
            for th in thetas:
                for f in self.index.get((l[0], l[1]), ()):
                    t2 = match(l, f, th)
                    if t2 is not None: new.append(t2)
            thetas = new
            if not thetas: return []
        out = []
        for th in thetas:
            vars_needed = {t for l in naf + [concl] for t in (l[2], l[3]) if is_var(t) and t not in th}
            bindings = [th]
            for v in vars_needed:
                bindings = [dict(b, **{v: e}) for b in bindings for e in self.entities]
            for b in bindings:
                if all(self.holds_naf(subst(l, b)) for l in naf):
                    g = subst(concl, b)
                    if any(is_var(t) for t in (g[2], g[3])): continue
                    if self.mode == "cwa" and g[0]: continue
                    out.append((g, [subst(l, b) for l in conds]))
        return out

    def run(self):
        st = self.strata()
        levels = sorted(set(st.values())) if st else []
        for lev in levels:
            rs = [r for r in self.rules if st[r[1][1]] == lev]
            for _ in range(self.max_iter):
                added = False
                for conds, concl in rs:
                    for g, body in self.fire(conds, concl):
                        self.steps += 1
                        if g not in self.facts:
                            self.facts.add(g); self.index[(g[0], g[1])].add(g)
                            self.trace[g] = ("rule", body); added = True
                if not added: break

    def query(self, lit):
        if self.mode == "cwa":
            if lit[0]: return self.holds_naf(lit)
            return lit in self.facts
        return lit in self.facts

def show_lit(l):
    p = l[1]
    s = ("NOT " if l[0] else "") + p.replace(":=", "").replace("be:", "").replace("can:", "can_")
    return f"{s}({l[2]}" + (f", {l[3]})" if l[3] else ")")

def explain(fc, lit, depth=0, seen=None):
    seen = seen or set()
    if lit not in fc.trace or lit in seen: return []
    seen.add(lit)
    t = fc.trace[lit]
    if t[0] == "given": return ["  " * depth + show_lit(lit) + "   [fact]"]
    lines = ["  " * depth + show_lit(lit) + "   <= " + " AND ".join(show_lit(b) for b in t[1])]
    for b in t[1]:
        if b in fc.facts: lines += explain(fc, b, depth + 1, seen)
    return lines

def ruletaker_predict(context, statement, mode="cwa", return_trace=False):
    facts, rules, unparsed, parser = build_kb(context)
    q = parser.parse_sentence(statement)
    fc = ForwardChainer(facts, rules, mode=mode)
    if q is None or q[0] != "fact":
        ans = False
    else:
        ans = fc.query(q[1])
    if return_trace:
        tr = explain(fc, q[1]) if (q and q[0] == "fact" and not q[1][0]) else []
        return ans, {"facts": facts, "rules": rules, "unparsed": unparsed,
                     "query": q, "trace": tr, "steps": fc.steps}
    return ans

## 4. RuleTaker (`jise/ruletaker` format): propositional rules + stratified forward chaining

In [4]:
# ============================================================
# Cell 2b - RuleTaker (jise/ruletaker format): propositional rules + forward chaining
# ============================================================
# jise/ruletaker contexts look like
#   "Facts: The wall dances gracefully. The river becomes famous. ...
#    Rules: Assuming the river dances gracefully, we can say the river is fast.
#           Based on charlie is fast, it is evident that stanford university volunteers.
#           Given that the globe is colorful, it follows that mike is colorful.
#           If the globe is friendly, then the dog runs marathons. ..."
# Facts and rules are ground (no variables), with many surface paraphrases of
# "if A then B". Each sentence becomes a propositional literal / Horn rule and the
# query is decided by stratified forward chaining with negation-as-failure.

ANTE_CUE = (r"(?:if|assuming(?:\s+that)?|given(?:\s+that)?|based\s+on(?:\s+the\s+fact\s+that)?|since|because|as\s+long\s+as|"
            r"when(?:ever)?|provided(?:\s+that)?|in\s+case|on\s+the\s+condition\s+that|considering(?:\s+that)?|"
            r"knowing(?:\s+that)?|due\s+to\s+the\s+fact\s+that|from\s+the\s+fact\s+that|in\s+light\s+of(?:\s+the\s+fact\s+that)?|"
            r"supposing(?:\s+that)?|suppose(?:\s+that)?|once|upon\s+knowing\s+that|as)")
CONS_CUE = re.compile(
    r"^(?:(?:we|one|you)\s+(?:can|could|may|might|must|should|will)\s+(?:safely\s+)?(?:say|conclude|infer|deduce|assume|state|claim|determine|tell)\s+(?:that\s+)?"
    r"|(?:we|one)\s+(?:know|see|find|get)\s+(?:that\s+)?"
    r"|it\s+(?:is|becomes|would\s+be)\s+(?:\w+\s+){0,2}?(?:evident|clear|obvious|apparent|certain|true|safe\s+to\s+say|reasonable\s+to\s+conclude|implied|established)\s+that\s+"
    r"|it\s+(?:follows|stands\s+to\s+reason|must\s+be\s+(?:the\s+case\s+)?|can\s+be\s+(?:said|concluded|inferred|deduced|assumed))\s+that\s+"
    r"|this\s+(?:means|implies|suggests|indicates|shows|entails|tells\s+us)\s+(?:that\s+)?"
    r"|(?:therefore|thus|hence|consequently|so|accordingly|as\s+a\s+result|it\s+follows)\s*,?\s+"
    r"|(?:then\s+)?it\s+is\s+the\s+case\s+that\s+)", re.I)
RULE_PATTERNS = [
    re.compile(rf"^{ANTE_CUE}\s+(?P<a>.+?)\s*,\s*(?P<c>.+)$", re.I),                       # If A, (then) B
    re.compile(rf"^{ANTE_CUE}\s+(?P<a>.+?)\s+then\s+(?P<c>.+)$", re.I),                    # If A then B
    re.compile(r"^(?P<a>.+?)\s+(?:implies|means|entails|leads\s+to|results\s+in)\s+(?:that\s+)?(?P<c>.+)$", re.I),
    re.compile(r"^(?P<a>.+?)\s*,\s*(?:therefore|thus|hence|so|then|consequently)\s*,?\s+(?P<c>.+)$", re.I),
    re.compile(r"^(?P<c>.+?)\s+(?:if|when|whenever|provided\s+that|given\s+that|as\s+long\s+as)\s+(?P<a>.+)$", re.I),  # B if A
]
NEG_WORDS = {"not", "never", "no"}
AUX_NEG = {"does", "do", "did"}
DROP_WORDS = {"the", "a", "an"}
PROP_STATS = Counter()          # parser coverage, reported after evaluation
PROP_UNPARSED = []

def _strip_cons(c):
    prev = None
    while prev != c:
        prev = c
        c = re.sub(r"^(?:then\b|,)\s*,?\s*", "", c, flags=re.I)
        c = CONS_CUE.sub("", c, count=1).strip()
    m = re.match(r"^(?:\w+\s+){0,5}?(?:say|evident|follows|conclude|concluded|means|implies|infer|deduce|clear|true)\s+(?:that\s+)?(.+)$", c, re.I)
    return m.group(1) if m else c

def prop_literal(text):
    """'The river does not dance gracefully' -> ('-', 'river dance gracefully')."""
    s = norm_sentence(text)                       # lower-case, "n't" -> " not"
    s = re.sub(r"^(?:the\s+fact\s+that|that)\s+", "", s)
    s = re.sub(r"\bcannot\b", "can not", s)
    words = re.findall(r"[a-z0-9]+", s)
    neg = any(w in NEG_WORDS for w in words)
    out = []
    for i, w in enumerate(words):
        if w in NEG_WORDS or w in DROP_WORDS: continue
        if neg and w in AUX_NEG and i + 1 < len(words) and words[i + 1] in NEG_WORDS: continue
        out.append(lemma(w))
    return ("-" if neg else "+", " ".join(out))

def _conj(text):
    parts = re.split(r"\s*,\s*and\s+|\s+and\s+|\s*,\s*", text.strip())
    return [prop_literal(p) for p in parts if p.strip()]

def parse_prop_rule(sent):
    s = sent.strip().rstrip(".").strip()
    for pat in RULE_PATTERNS:
        m = pat.match(s)
        if not m: continue
        a, c = m.group("a").strip(), _strip_cons(m.group("c").strip())
        if not a or not c: continue
        heads = _conj(c)
        rules = []
        for disj in re.split(r"\s+or\s+", a):        # (A or B) => C  ==  A=>C, B=>C
            body = _conj(disj)
            if body and heads:
                rules += [(tuple(body), h) for h in heads]
        if rules: return rules
    return None

def build_prop_kb(context):
    text = context.strip()
    m = re.search(r"\brules?\s*:", text, re.I)
    fact_txt, rule_txt = (text[:m.start()], text[m.end():]) if m else (text, "")
    fact_txt = re.sub(r"^\s*facts?\s*:", "", fact_txt, flags=re.I)
    facts, rules, unparsed = set(), [], []
    for s in split_sentences(fact_txt):
        r = parse_prop_rule(s) if not m else None      # without headers, rules may sit among facts
        if r: rules += r
        else:
            for lit in _conj(s.rstrip(".")): facts.add(lit)
    for s in split_sentences(rule_txt):
        PROP_STATS["rule_sentences"] += 1
        r = parse_prop_rule(s)
        if r: rules += r
        else:
            unparsed.append(s); PROP_STATS["unparsed"] += 1
            if len(PROP_UNPARSED) < 20: PROP_UNPARSED.append(s)
    return facts, rules, unparsed

def prop_forward_chain(facts, rules, mode="cwa"):
    """Stratified forward chaining over ground literals; returns (model, justification, steps)."""
    atoms = {l[1] for l in facts} | {b[1] for r in rules for b in r[0]} | {r[1][1] for r in rules}
    level = {a: 0 for a in atoms}
    for _ in range(len(atoms) + 1):                     # stratify (NAF only in CWA)
        changed = False
        for body, h in rules:
            need = max([level[b[1]] + (1 if (b[0] == "-" and mode == "cwa") else 0) for b in body] + [0])
            if level[h[1]] < need and need <= len(atoms):
                level[h[1]] = need; changed = True
        if not changed: break
    F = set(facts); why = {f: None for f in facts}; steps = 0
    for k in sorted(set(level.values())):
        layer = [r for r in rules if level[r[1][1]] == k]
        changed = True
        while changed:
            changed = False
            for body, h in layer:
                if h in F: continue
                ok = True
                for b in body:
                    if b[0] == "+": ok = b in F
                    elif mode == "cwa": ok = b in F or ("+", b[1]) not in F
                    else: ok = b in F
                    if not ok: break
                if ok:
                    F.add(h); why[h] = body; changed = True; steps += 1
    return F, why, steps

def _show(l): return ("NOT " if l[0] == "-" else "") + l[1]

def _explain_prop(why, lit, depth=0, seen=None):
    seen = seen or set()
    if lit not in why or lit in seen: return []
    seen.add(lit)
    body = why[lit]
    if body is None: return ["  " * depth + f"{_show(lit)}  [fact]"]
    out = ["  " * depth + f"{_show(lit)}  <= " + " AND ".join(_show(b) for b in body)]
    for b in body:
        out += _explain_prop(why, b, depth + 1, seen) if b in why else ["  " * (depth + 1) + f"{_show(b)}  [NAF]"]
    return out

def prop_ruletaker_predict(context, statement, mode="cwa", return_trace=False):
    facts, rules, unparsed = build_prop_kb(context)
    q = prop_literal(statement.rstrip("."))
    F, why, steps = prop_forward_chain(facts, rules, mode)
    if q[0] == "+": ans = q in F
    elif mode == "cwa": ans = q in F or ("+", q[1]) not in F
    else: ans = q in F
    if return_trace:
        tr = _explain_prop(why, q) if q in why else [f"{_show(q)}: " + ("not derivable (closed world)" if not ans else "holds by negation-as-failure")]
        return ans, {"query": _show(q), "trace": tr, "steps": steps,
                     "n_facts": len(facts), "n_rules": len(rules), "unparsed": unparsed}
    return ans

def is_prop_ruletaker(context):
    return bool(re.match(r"\s*facts?\s*:", context, re.I) or re.search(r"\brules?\s*:", context, re.I))

## 5. FOLIO: FOL parser, CNF conversion, resolution refutation

In [5]:
FOL_SYMS = [("<->", "↔"), ("->", "→"), ("⇒", "→"), ("⇔", "↔"), ("&", "∧"), ("|", "∨"),
            ("~", "¬"), ("!", "¬"), ("≠", " ≠ "), ("⊕", " ⊕ "), ("forall ", "∀"), ("exists ", "∃")]

def fol_tokenize(s):
    for a, b in FOL_SYMS: s = s.replace(a, b)
    return re.findall(r"[∀∃¬∧∨→↔⊕()=,]|≠|[A-Za-z0-9_'\.\-]+", s)

class FOLParser:
    PREC = {"↔": 1, "→": 2, "⊕": 3, "∨": 4, "∧": 5}
    OPS = {"↔": "iff", "→": "imp", "⊕": "xor", "∨": "or", "∧": "and"}

    def parse(self, s):
        self.toks, self.i = fol_tokenize(s), 0
        f = self.expr(set(), 0)
        return f

    def peek(self): return self.toks[self.i] if self.i < len(self.toks) else None
    def take(self):
        t = self.peek(); self.i += 1; return t

    def expr(self, bound, minp):
        left = self.unary(bound)
        while True:
            op = self.peek()
            if op not in self.PREC or self.PREC[op] < minp: break
            self.take()
            p = self.PREC[op]
            right = self.expr(bound, p + 1 if op != "→" else p)   # → right-assoc
            left = (self.OPS[op], left, right)
        return left

    def unary(self, bound):
        t = self.peek()
        if t == "¬":
            self.take(); return ("not", self.unary(bound))
        if t in ("∀", "∃"):
            self.take(); v = self.take()
            # allow "∀x ∀y" and "∀x,y"
            vs = [v]
            while self.peek() == ",":
                self.take(); vs.append(self.take())
            nb = bound | set(vs)
            body = self.expr(nb, 0) if self.peek() not in ("∀", "∃") else self.unary(nb)
            for v in reversed(vs):
                body = ("all" if t == "∀" else "ex", v, body)
            return body
        if t == "(":
            self.take(); f = self.expr(bound, 0)
            if self.peek() == ")": self.take()
            return f
        return self.atom(bound)

    def term(self, bound):
        name = self.take()
        if self.peek() == "(":
            self.take(); args = []
            while self.peek() not in (")", None):
                args.append(self.term(bound))
                if self.peek() == ",": self.take()
            self.take()
            return ("f", name, tuple(args))
        return ("v", name) if name in bound else ("c", name.lower())

    def atom(self, bound):
        name = self.take()
        if name is None: raise ValueError("unexpected end")
        args = []
        if self.peek() == "(":
            self.take()
            while self.peek() not in (")", None):
                args.append(self.term(bound))
                if self.peek() == ",": self.take()
            self.take()
        else:
            # bare term followed by = / ≠
            if self.peek() in ("=", "≠"):
                lhs = ("v", name) if name in bound else ("c", name.lower())
                op = self.take(); rhs = self.term(bound)
                a = ("atom", "=", (lhs, rhs))
                return a if op == "=" else ("not", a)
        a = ("atom", name, tuple(args))
        if self.peek() in ("=", "≠") and not args:
            pass
        return a

# ---------- CNF conversion ----------
def elim(f):
    k = f[0]
    if k == "atom": return f
    if k == "not": return ("not", elim(f[1]))
    if k in ("all", "ex"): return (k, f[1], elim(f[2]))
    a, b = elim(f[1]), elim(f[2])
    if k == "imp": return ("or", ("not", a), b)
    if k == "iff": return ("and", ("or", ("not", a), b), ("or", a, ("not", b)))
    if k == "xor": return ("and", ("or", a, b), ("or", ("not", a), ("not", b)))
    return (k, a, b)

def nnf(f, neg=False):
    k = f[0]
    if k == "atom": return ("not", f) if neg else f
    if k == "not": return nnf(f[1], not neg)
    if k in ("and", "or"):
        kk = k if not neg else ("or" if k == "and" else "and")
        return (kk, nnf(f[1], neg), nnf(f[2], neg))
    if k in ("all", "ex"):
        kk = k if not neg else ("ex" if k == "all" else "all")
        return (kk, f[1], nnf(f[2], neg))
    raise ValueError(k)

_ctr = itertools.count()

def subst_term(t, m):
    if t[0] == "v": return m.get(t[1], t)
    if t[0] == "f": return ("f", t[1], tuple(subst_term(a, m) for a in t[2]))
    return t

def skolem(f, univ, m):
    k = f[0]
    if k == "atom": return ("atom", f[1], tuple(subst_term(a, m) for a in f[2]))
    if k == "not": return ("not", skolem(f[1], univ, m))
    if k in ("and", "or"): return (k, skolem(f[1], univ, m), skolem(f[2], univ, m))
    if k == "all":
        nv = ("v", f"V{next(_ctr)}")
        return skolem(f[2], univ + [nv], dict(m, **{f[1]: nv}))
    if k == "ex":
        n = next(_ctr)
        sk = ("f", f"sk{n}", tuple(univ)) if univ else ("c", f"sk{n}")
        return skolem(f[2], univ, dict(m, **{f[1]: sk}))
    raise ValueError(k)

def to_clauses(f, limit=2000):
    k = f[0]
    if k == "and":
        return to_clauses(f[1], limit) + to_clauses(f[2], limit)
    if k == "or":
        A, B = to_clauses(f[1], limit), to_clauses(f[2], limit)
        if len(A) * len(B) > limit: raise ValueError("CNF blow-up")
        return [a | b for a in A for b in B]
    if k == "not": return [frozenset([(False, f[1][1], f[1][2])])]
    return [frozenset([(True, f[1], f[2])])]

def is_taut(c):
    return any((not s, p, a) in c for (s, p, a) in c)

def formula_to_clauses(f):
    cl = to_clauses(skolem(nnf(elim(f)), [], {}))
    return [c for c in cl if not is_taut(c)]

# ---------- Unification and resolution ----------
def walk(t, th):
    while t[0] == "v" and t[1] in th: t = th[t[1]]
    return t

def occurs(v, t, th):
    t = walk(t, th)
    if t[0] == "v": return t[1] == v
    if t[0] == "f": return any(occurs(v, a, th) for a in t[2])
    return False

def unify(a, b, th):
    a, b = walk(a, th), walk(b, th)
    if a == b: return th
    if a[0] == "v":
        if occurs(a[1], b, th): return None
        return dict(th, **{a[1]: b})
    if b[0] == "v": return unify(b, a, th)
    if a[0] == "f" and b[0] == "f" and a[1] == b[1] and len(a[2]) == len(b[2]):
        for x, y in zip(a[2], b[2]):
            th = unify(x, y, th)
            if th is None: return None
        return th
    return None

def unify_args(xs, ys, th):
    if len(xs) != len(ys): return None
    for x, y in zip(xs, ys):
        th = unify(x, y, th)
        if th is None: return None
    return th

def resolve_term(t, th):
    t = walk(t, th)
    if t[0] == "f": return ("f", t[1], tuple(resolve_term(a, th) for a in t[2]))
    return t

def rename(c):
    m = {}
    def r(t):
        if t[0] == "v":
            if t[1] not in m: m[t[1]] = ("v", f"R{next(_ctr)}")
            return m[t[1]]
        if t[0] == "f": return ("f", t[1], tuple(r(a) for a in t[2]))
        return t
    return frozenset((s, p, tuple(r(a) for a in args)) for (s, p, args) in c)

def term_depth(t):
    return 1 + max((term_depth(a) for a in t[2]), default=0) if t[0] == "f" else 0

def clause_weight(c):
    return len(c) * 10 + sum(term_depth(a) for (_, _, args) in c for a in args)

def canon(c):
    """Canonical form (variables renamed by first occurrence) for duplicate detection."""
    m = {}
    def r(t):
        if t[0] == "v":
            if t[1] not in m: m[t[1]] = ("v", f"_{len(m)}")
            return m[t[1]]
        if t[0] == "f": return ("f", t[1], tuple(r(a) for a in t[2]))
        return t
    lits = sorted(c, key=repr)
    return frozenset((s, p, tuple(r(a) for a in args)) for (s, p, args) in lits)

def resolvents(c1, c2):
    out = []
    c2 = rename(c2)
    for (s1, p1, a1) in c1:
        for (s2, p2, a2) in c2:
            if p1 == p2 and s1 != s2:
                th = unify_args(a1, a2, {})
                if th is None: continue
                new = set()
                for (s, p, a) in c1:
                    if (s, p, a) != (s1, p1, a1): new.add((s, p, tuple(resolve_term(x, th) for x in a)))
                for (s, p, a) in c2:
                    if (s, p, a) != (s2, p2, a2): new.add((s, p, tuple(resolve_term(x, th) for x in a)))
                nc = frozenset(new)
                if not is_taut(nc): out.append(nc)
    return out

def factor(c):
    """Simple factoring: merge pairs of unifiable same-sign literals."""
    out = [c]
    lits = list(c)
    for i in range(len(lits)):
        for j in range(i + 1, len(lits)):
            a, b = lits[i], lits[j]
            if a[0] == b[0] and a[1] == b[1]:
                th = unify_args(a[2], b[2], {})
                if th is not None:
                    out.append(frozenset((s, p, tuple(resolve_term(x, th) for x in ar)) for (s, p, ar) in c))
    return out

def resolution_prove(kb_clauses, goal_clauses, time_limit=2.0, max_clauses=4000):
    """Set-of-support, given-clause resolution. Returns (proved, info)."""
    import heapq
    t0 = time.time()
    processed = list(kb_clauses)
    seen = set(canon(c) for c in kb_clauses)
    heap = []
    for c in goal_clauses:
        heapq.heappush(heap, (clause_weight(c), next(_ctr), c))
        seen.add(canon(c))
    steps = 0
    by_pred = defaultdict(list)
    for c in processed:
        for (s, p, _) in c: by_pred[(not s, p)].append(c)
    while heap:
        if time.time() - t0 > time_limit or len(seen) > max_clauses:
            return False, {"status": "timeout", "steps": steps}
        _, _, given = heapq.heappop(heap)
        cands = []
        for (s, p, _) in given:
            cands.extend(by_pred.get((s, p), ()))
        for other in set(cands) | {given}:
            for r in resolvents(given, other):
                steps += 1
                if not r: return True, {"status": "proved", "steps": steps}
                for rf in factor(r):
                    k = canon(rf)
                    if k in seen or len(rf) > 8: continue
                    seen.add(k)
                    heapq.heappush(heap, (clause_weight(rf), next(_ctr), rf))
        processed.append(given)
        for (s, p, _) in given: by_pred[(not s, p)].append(given)
    return False, {"status": "saturated", "steps": steps}

_fol_parser = FOLParser()

def fol_entails(premises_fol, conclusion_fol, time_limit=None, max_clauses=None):
    time_limit = time_limit or PROVER_TIME_LIMIT
    max_clauses = max_clauses or PROVER_MAX_CLAUSES
    kb = []
    for p in premises_fol:
        p = p.strip()
        if not p: continue
        try: kb += formula_to_clauses(_fol_parser.parse(p))
        except Exception: pass
    goal = _fol_parser.parse(conclusion_fol)
    neg_goal = formula_to_clauses(("not", goal))
    return resolution_prove(kb, neg_goal, time_limit, max_clauses)

def split_fol_premises(x):
    if isinstance(x, (list, tuple)): return list(x)
    return [l for l in str(x).split("\n") if l.strip()]

def folio_predict(premises_fol, conclusion_fol, return_trace=False):
    try:
        ok, info = fol_entails(split_fol_premises(premises_fol), conclusion_fol)
    except Exception as e:
        ok, info = False, {"status": f"parse-error: {e}"}
    return (ok, info) if return_trace else ok

## 6. CounterBench: causal model + counterfactual search

In [6]:
CAUSE_RE = r"(?:causes|cause|caused|leads to|lead to|results in|result in|produces|produce|triggers|trigger)"
PREVENT_RE = r"(?:prevents|prevent|inhibits|inhibit|blocks|block|stops|stop)"

def _clean_var(t):
    t = t.strip().strip(",.;:?").lower()
    t = re.sub(r"^(?:the occurrence of|the presence of|that)\s+", "", t)
    t = re.sub(r"\s+(?:occurs|occurring|happens|occur|happen|together|jointly)$", "", t)
    return t.strip()

def _bexpr(text):
    """'not a and b' / 'a or b' / 'a, b and c together' -> nested tuple expression."""
    text = text.strip().strip(",.").lower()
    text = re.sub(r"\b(?:together|jointly)\b", "", text)
    text = re.sub(r"^(?:either|both)\s+", "", text).strip()
    def lit(t):
        t = t.strip()
        neg = False
        m = re.match(r"^(?:not|no)\s+(.+)$", t)
        if m: neg, t = True, m.group(1)
        v = ("var", _clean_var(t))
        return ("not", v) if neg else v
    def conj(t):
        parts = [p for p in re.split(r"\s+and\s+|,\s*", t) if p.strip()]
        e = lit(parts[0])
        for p in parts[1:]: e = ("and", e, lit(p))
        return e
    ors = [t for t in re.split(r"\s+or\s+", text) if t.strip()]
    e = conj(ors[0])
    for o in ors[1:]: e = ("or", e, conj(o))
    return e

def _eval(e, val):
    k = e[0]
    if k == "var": return val.get(e[1], False)
    if k == "not": return not _eval(e[1], val)
    if k == "and": return _eval(e[1], val) and _eval(e[2], val)
    return _eval(e[1], val) or _eval(e[2], val)

def _vars(e, acc):
    if e[0] == "var": acc.add(e[1])
    else:
        for x in e[1:]: _vars(x, acc)
    return acc

def split_relations(text):
    """Split 'A causes B, B and C together cause D, and D causes E' into relations.
    Comma-fragments without a causal verb are merged into the next fragment
    (handles 'A, B and C together cause D')."""
    out = []
    for sent in re.split(r"(?<=[.;])\s+|\n+", text):
        sent = sent.strip().rstrip(".;")
        if not sent: continue
        frags = re.split(r",\s*(?:and\s+)?", sent)
        buf = ""
        for f in frags:
            buf = (buf + ", " + f) if buf else f
            if re.search(rf"\b{CAUSE_RE}\b|\b{PREVENT_RE}\b|\bobserved?\b|\boccurs?\b|\bhappen", buf):
                out.append(buf.strip()); buf = ""
        if buf: out.append(buf.strip())
    return out

class CausalModel:
    def __init__(self):
        self.eq = defaultdict(list)   # var -> list of expressions (OR-combined)
        self.obs = {}

    def add_relation(self, s):
        s = s.strip().rstrip(".").lower()
        s = re.sub(r"^(?:we know that|it is known that|suppose that|assume that|imagine that|"
                   r"we also know that|also,?|and)\s+", "", s).strip()
        m = re.match(rf"^(.+?)\s+{PREVENT_RE}\s+(.+)$", s)
        if m:
            self.eq[_clean_var(m.group(2))].append(("not", _bexpr(m.group(1)))); return True
        m = re.match(rf"^(.+?)\s+{CAUSE_RE}\s+(not\s+)?(.+)$", s)
        if m:
            e = _bexpr(m.group(1))
            self.eq[_clean_var(m.group(3))].append(("not", e) if m.group(2) else e); return True
        m = re.match(r"^(?:we\s+)?(?:observe|observed|see|saw|notice|noticed)(?:\s+that)?\s+(.+)$", s) or \
            re.match(r"^it (?:is|was) observed that\s+(.+)$", s)
        if m:
            for part in re.split(r"\s+and\s+|,\s*", m.group(1)):
                part = part.strip()
                neg = bool(re.match(r"^(?:not|no)\s+", part)) or bool(re.search(r"(?:does|did) not (?:occur|happen)", part))
                part = re.sub(r"^(?:not|no)\s+", "", part)
                part = re.sub(r"\s+(?:does|did) not (?:occur|happen)$", "", part)
                if part: self.obs[_clean_var(part)] = not neg
            return True
        m = re.match(r"^(.+?)\s+(?:does not|did not|doesn't|didn't) (?:occur|happen)$", s)
        if m: self.obs[_clean_var(m.group(1))] = False; return True
        m = re.match(r"^(.+?)\s+(?:occurs|occurred|happens|happened|is true|is present)$", s)
        if m: self.obs[_clean_var(m.group(1))] = True; return True
        return False

    def variables(self):
        vs = set(self.eq) | set(self.obs)
        for es in self.eq.values():
            for e in es: _vars(e, vs)
        return vs

    def topo(self):
        deps = {v: (set().union(*[_vars(e, set()) for e in self.eq[v]]) if v in self.eq else set())
                for v in self.variables()}
        order, done = [], set()
        def visit(v, stack=()):
            if v in done or v in stack: return
            for d in deps.get(v, ()): visit(d, stack + (v,))
            done.add(v); order.append(v)
        for v in sorted(deps): visit(v)
        return order

    def compute(self, exo, do=None):
        do = do or {}
        val = dict(exo); val.update(do)
        for v in self.topo():
            if v in do: continue
            if v in self.eq: val[v] = any(_eval(e, val) for e in self.eq[v])
        return val

    def abduce(self, factual):
        """Exogenous assignment consistent with evidence; among consistent worlds,
        prefer the one closest to the default (unobserved exogenous absent)."""
        exo = sorted(v for v in self.variables() if v not in self.eq)
        evidence = dict(self.obs); evidence.update(factual)
        base = {v: evidence.get(v, False) for v in exo}
        if all(self.compute(base).get(k) == v for k, v in evidence.items()):
            return base, []
        free = [v for v in exo if v not in evidence]
        best = None
        if len(free) <= 16:
            for k in range(len(free) + 1):          # breadth-first: fewest changes first
                for flip in itertools.combinations(free, k):
                    w = dict(base); w.update({v: True for v in flip})
                    if all(self.compute(w).get(kk) == vv for kk, vv in evidence.items()):
                        best = w; break
                if best: break
        if best: return best, []
        # evidence on endogenous variables that no world explains: hold them fixed
        fixed = {k: v for k, v in evidence.items() if k in self.eq}
        return base, fixed

    def query(self, target, do, factual):
        world, fixed = self.abduce(factual)
        d = dict(fixed); d.update(do)
        return self.compute(world, d).get(target, False), world

def _interventions(text):
    """'not ziklo and not vork' -> {'ziklo': False, 'vork': False}"""
    out = {}
    for part in re.split(r"\s+and\s+|,\s*", text):
        part = part.strip()
        if not part: continue
        neg = bool(re.match(r"^(?:not|no)\s+", part))
        part = re.sub(r"^(?:not|no)\s+", "", part)
        part = re.sub(r"\s+(?:had\s+)?(?:occurred|happened|occur|happen)$", "", part)
        v = _clean_var(part)
        if v: out[v] = not neg
    return out

def parse_cf_question(q):
    """Return (target, negated_target, interventions, factual) or None."""
    q = re.sub(r"\s+", " ", q.strip().lower())
    do, factual = {}, {}
    # nested: "assume not X ... further suppose not V. would Y occur?"
    for m in re.finditer(r"\b(?:assume|suppose|imagine|if we set)(?: that)?\s+([^.?]+?)(?=[.?]|,? (?:further|then|and further|would))", q):
        do.update(_interventions(m.group(1)))
    m = re.search(r"would\s+(not\s+)?(.+?)\s+(?:occur|happen|be true|have occurred)\b(?:\s+if\s+(.+?))?\s*\??$", q)
    if not m: return None
    tneg, target, cond = m.group(1), _clean_var(m.group(2)), m.group(3)
    if cond:
        mi = re.match(r"^(.+?)\s+instead of\s+(.+)$", cond)
        if mi:
            do.update(_interventions(mi.group(1)))
            factual.update(_interventions(mi.group(2)))
        else:
            do.update(_interventions(cond))
    return target, bool(tneg), do, factual

def counterbench_predict(context, question, return_trace=False):
    cm = CausalModel()
    unparsed = []
    full = (context or "").strip()
    qtext = (question or "").strip()
    # if everything is in one field, separate the "would ..." question part
    if not qtext or not re.search(r"\bwould\b", qtext.lower()):
        m = re.search(r"(?i)((?:assume|suppose)[^?]*?)?would[^?]*\?", full)
        if m: qtext, full = m.group(0), full[:m.start()]
    # premises that live in the question (e.g. nested "Assume not X, ...") are handled by the question parser
    for rel in split_relations(full):
        if not cm.add_relation(rel): unparsed.append(rel)
    ans, info = False, {}
    pq = parse_cf_question(qtext)
    if pq:
        target, tneg, do, factual = pq
        res, world = cm.query(target, do, factual)
        ans = (not res) if tneg else res
        info = {"target": target, "do": do, "factual": factual, "world": world}
    if return_trace:
        info.update({"equations": {k: v for k, v in cm.eq.items()}, "observations": cm.obs,
                     "unparsed": unparsed})
        return ans, info
    return ans

## 7. Row normalisation and router
Every row is mapped to one dict (`context`, `statement`, FOL fields if present, `label`). The router picks the engine:
FOL annotation → resolution; causal/counterfactual question → CounterBench engine; `Facts:`/`Rules:` → propositional RuleTaker engine; otherwise the generic RuleTaker parser.

In [7]:
CTX_KEYS = ["context", "given_info", "premises", "theory", "facts", "story", "passage", "kb"]
Q_KEYS = ["statement", "text", "question", "conclusion", "query", "hypothesis"]
LABEL_KEYS = ["label", "answer", "flag", "gold", "gold_label", "output", "target"]
PFOL_KEYS = ["premises-FOL", "premises_fol", "premises_FOL", "premise_fol"]
CFOL_KEYS = ["conclusion-FOL", "conclusion_fol", "conclusion_FOL"]

def pick(ex, keys):
    for k in keys:
        if k in ex and ex[k] not in (None, ""): return ex[k]
    return None

def to_bool(v):
    if isinstance(v, bool): return v
    if isinstance(v, (int, float)): return bool(v)
    s = str(v).strip().lower()
    if s in ("true", "yes", "entailment", "entailed", "1", "t", "y"): return True
    return False   # false / no / not entailment / uncertain / unknown / contradiction

def as_text(x):
    return "\n".join(map(str, x)) if isinstance(x, (list, tuple)) else ("" if x is None else str(x))

def normalize(ex, dataset):
    return {
        "id": ex.get("id", ex.get("example_id", ex.get("question_id", ex.get("story_id")))),
        "dataset": dataset,
        "context": as_text(pick(ex, CTX_KEYS)),
        "statement": as_text(pick(ex, Q_KEYS)),
        "premises_fol": pick(ex, PFOL_KEYS),
        "conclusion_fol": pick(ex, CFOL_KEYS),
        "label": to_bool(pick(ex, LABEL_KEYS)) if pick(ex, LABEL_KEYS) is not None else None,
        "config": ex.get("config") or ex.get("type") or
                  (f"depth-{ex['depth']}" if ex.get("depth") is not None else None),
        "split": ex.get("split"),
    }

def _hf_split_to_list(ds):
    feats = getattr(ds, "features", {}) or {}
    rows = []
    for ex in ds:
        ex = dict(ex)
        for k, f in feats.items():
            if type(f).__name__ == "ClassLabel" and isinstance(ex.get(k), int):
                ex[k] = f.int2str(ex[k])
        rows.append(ex)
    return rows

RULE_MODE = "cwa"   # chosen on the RuleTaker train split in Section 8

def is_counterfactual(text):
    return bool(re.search(r"\bwould\b.+\bif\b.+\binstead of\b", text.lower()))

def is_causal(ex):
    if ex.get("dataset") == "counterbench": return True
    q = (ex["statement"] + " " + ex["context"]).lower()
    return bool(re.search(r"\bwould\b[^?]*\b(?:occur|happen)\b", q) and re.search(r"\bcauses?\b", q))

def predict(ex, return_trace=False):
    """Route an example to the appropriate classical reasoner."""
    if ex.get("conclusion_fol"):
        return folio_predict(ex["premises_fol"] or "", ex["conclusion_fol"], return_trace)
    if is_causal(ex) or is_counterfactual(ex["statement"]):
        return counterbench_predict(ex["context"], ex["statement"], return_trace)
    if is_prop_ruletaker(ex["context"]):          # jise/ruletaker "Facts: ... Rules: ..." format
        return prop_ruletaker_predict(ex["context"], ex["statement"], RULE_MODE, return_trace)
    return ruletaker_predict(ex["context"], ex["statement"], RULE_MODE, return_trace)

## 8. Results on the public data (accuracy + inference time)
All public samples are used: RuleTaker train + test + ood_test, FOLIO train + validation, both CounterBench question files.
The only data-driven choice (closed- vs open-world negation for RuleTaker) is made on the RuleTaker **train** split;
accuracy is also printed per split, so the RuleTaker test / ood_test numbers are clean.

**Overall Inference Time** = average time per sample in each dataset, averaged over the three datasets.

In [8]:
def evaluate(examples, limit=None, desc=""):
    if limit and len(examples) > limit:
        examples = random.Random(SEED).sample(examples, limit)
    correct, n, per_cfg = 0, 0, defaultdict(lambda: [0, 0])
    per_split = defaultdict(lambda: [0, 0])
    preds = []
    t0 = time.perf_counter()
    for ex in examples:
        p = predict(ex)
        preds.append(p)
        if ex["label"] is not None:
            n += 1; ok = (p == ex["label"]); correct += ok
            c = per_cfg[str(ex.get("config"))]; c[0] += ok; c[1] += 1
            c = per_split[str(ex.get("split"))]; c[0] += ok; c[1] += 1
    dt = time.perf_counter() - t0
    acc = 100.0 * correct / n if n else float("nan")
    print(f"{desc:<28} n={n:<6} accuracy={acc:6.2f}%  time={dt:8.2f}s  ({1000*dt/max(1,len(examples)):.2f} ms/query)")
    n_samples = len(examples)
    return {"n": n, "n_samples": n_samples, "accuracy": acc, "time_sec": dt,
            "avg_time_per_sample_sec": dt / max(1, n_samples),
            "per_config": {k: 100.0 * v[0] / v[1] for k, v in per_cfg.items() if v[1]},
            "per_split": {k: {"n": v[1], "accuracy": 100.0 * v[0] / v[1]} for k, v in per_split.items() if v[1]}}, preds


def rows_from(ds_dict, dataset):
    """{split_name: hf_dataset} -> list of normalised rows, tagged with their split."""
    out = []
    for split, d in ds_dict.items():
        for i, ex in enumerate(_hf_split_to_list(d)):
            ex["split"] = split
            if dataset == "counterbench":       # question ids repeat across files
                tag = split.replace("data_balanced_", "").replace(".json", "")
                ex["id"] = f"{tag}_{ex.get('question_id', i)}"
            out.append(normalize(ex, dataset))
    return out

if RUN_EXPERIMENTS:
    DATA = {
        "ruletaker":    rows_from({s: ds_rule_taker[s] for s in ds_rule_taker}, "ruletaker"),
        "folio":        rows_from({s: ds_folio[s] for s in ds_folio}, "folio"),
        "counterbench": rows_from(ds_counter_bench, "counterbench"),
    }
    for k, v in DATA.items(): print(f"{k:<13} {len(v)} samples")

    # --- choose the negation semantics (CWA / OWA) on the RuleTaker train split only
    dev = [e for e in DATA["ruletaker"] if e["split"] == "train"]
    scores = {}
    for m in ("cwa", "owa"):
        RULE_MODE = m
        scores[m] = evaluate(dev, desc=f"ruletaker train [{m}]")[0]["accuracy"]
    RULE_MODE = max(scores, key=scores.get)
    print("Selected negation semantics:", RULE_MODE)
    print("=" * 70)

    RESULTS = {"rule_mode": RULE_MODE, "datasets": {}}
    total_t, total_c, total_n = 0.0, 0, 0
    per_sample_times = []
    for name, rows in DATA.items():
        r, _ = evaluate(rows, desc=name)
        RESULTS["datasets"][name] = r
        total_t += r["time_sec"]; total_n += r["n"]; total_c += r["accuracy"] * r["n"] / 100
        per_sample_times.append(r["avg_time_per_sample_sec"])
        if len(r["per_split"]) > 1:
            for s, v in r["per_split"].items():
                print(f"    split {s:<32} n={v['n']:<6} accuracy={v['accuracy']:6.2f}%")
        for g, acc in sorted(r["per_config"].items()):
            if g != "None": print(f"    {g:<38} accuracy={acc:6.2f}%")

    RESULTS["overall_accuracy"] = 100.0 * total_c / total_n
    RESULTS["total_inference_time_sec"] = total_t
    RESULTS["overall_inference_time_sec"] = sum(per_sample_times) / len(per_sample_times)
    print("=" * 70)
    print(f"FINAL ACCURACY (all {total_n} public samples): {RESULTS['overall_accuracy']:.2f}%")
    print(f"TOTAL INFERENCE TIME: {total_t:.2f} s")
    for name, r in RESULTS["datasets"].items():
        print(f"  {name:<14} avg time per sample: {r['avg_time_per_sample_sec']:.6f} s")
    print(f"OVERALL INFERENCE TIME (average of the per-dataset averages): "
          f"{RESULTS['overall_inference_time_sec']:.6f} s per sample")

    if PROP_STATS["rule_sentences"]:
        print(f"RuleTaker rule parser: {PROP_STATS['rule_sentences']} rule sentences seen, "
              f"{PROP_STATS['unparsed']} unparsed")
        for s in PROP_UNPARSED[:10]: print("   unparsed:", s)
    RESULTS["ruletaker_rule_parser"] = {"rule_sentences": PROP_STATS["rule_sentences"],
                                        "unparsed": PROP_STATS["unparsed"], "examples": PROP_UNPARSED[:20]}
    with open("results.json", "w") as f: json.dump(RESULTS, f, indent=2, default=str)
    print("Saved results.json")

ruletaker     1625 samples
folio         1204 samples
counterbench  1200 samples
ruletaker train [cwa]        n=1000   accuracy= 98.30%  time=    0.41s  (0.41 ms/query)
ruletaker train [owa]        n=1000   accuracy= 98.30%  time=    0.36s  (0.36 ms/query)
Selected negation semantics: cwa
ruletaker                    n=1625   accuracy= 98.46%  time=    0.73s  (0.45 ms/query)
    split train                            n=1000   accuracy= 98.30%
    split test                             n=250    accuracy= 97.60%
    split ood_test                         n=375    accuracy= 99.47%
    depth-1                                accuracy=100.00%
    depth-2                                accuracy= 96.40%
    depth-3                                accuracy= 99.00%
    depth-4                                accuracy=100.00%
    depth-5                                accuracy= 98.40%
folio                        n=1204   accuracy= 81.15%  time=  135.59s  (112.62 ms/query)
    split train          

## 9. Example reasoning traces

In [9]:
if RUN_EXPERIMENTS:
    # --- example reasoning traces for the report
    TRACES = []
    for name, d in DATA.items():
        if not d: continue
        for ex in d[:3]:
            p, info = predict(ex, return_trace=True)
            TRACES.append({"dataset": name, "context": ex["context"][:600], "statement": ex["statement"],
                           "gold": ex["label"], "pred": p,
                           "trace": info.get("trace") or {k: str(v)[:300] for k, v in info.items()
                                                          if k not in ("facts", "rules")}})
    # accuracy/time per dataset and example proof traces (also used to build report.pdf)
    with open("traces.json", "w") as f: json.dump(TRACES, f, indent=2, default=str)
    print("Saved traces.json")
    for t in TRACES[:9]:
        print(f"[{t['dataset']}] gold={t['gold']} pred={t['pred']} | {t['statement']}")
        tr_ = t["trace"]
        print("   " + ("\n   ".join(map(str, tr_)) if isinstance(tr_, list) else str(tr_)))

Saved traces.json
[ruletaker] gold=False pred=False | The sandwich dances gracefully.
   sandwich dance gracefully: not derivable (closed world)
[ruletaker] gold=False pred=False | The river volunteers.
   river volunteer: not derivable (closed world)
[ruletaker] gold=True pred=True | Mike is colorful.
   mike be colorful  <= globe be colorful
     globe be colorful  [fact]
[folio] gold=True pred=True | Rina doesn't want to be addicted to caffeine or is unaware that caffeine is a drug.
   {'status': 'proved', 'steps': '9'}
[folio] gold=True pred=False | Rina eith doesn't want to be addicted to caffeine or is unaware that caffeine is a drug.
   {'status': 'saturated', 'steps': '132'}
[folio] gold=False pred=False | Rina either regularly drinks coffee or is unaware that caffeine is a drug.
   {'status': 'saturated', 'steps': '84'}
[counterbench] gold=False pred=False | Would Lumbo occur if not Ziklo instead of Ziklo?
   {'target': 'lumbo', 'do': "{'ziklo': False}", 'factual': "{'ziklo': 

## 10. Generate `submission.csv`
Set `TEST_PATH` to the test file (`.csv`, `.tsv`, `.json`, `.jsonl` or `.parquet`) and run this cell. Columns are found by name
(context: `context` / `given_info` / `premises`; statement: `statement` / `question` / `conclusion`; optional `premises-FOL` / `conclusion-FOL`;
id: `id` / `question_id` / `example_id`, otherwise the row number). It writes `id,label` and prints the row count, unreadable rows and the inference time.
If the test file is not found, predictions for all public samples are written instead.

In [10]:
import pandas as pd

# ================== the only line to change when the test file arrives ==================
TEST_PATH = "test.csv"            # .csv / .tsv / .json / .jsonl / .parquet
OUT_PATH  = "submission.csv"
# =========================================================================================

ID_COLS = ["id", "question_id", "example_id", "idx", "index"]

def load_test(path):
    if path.endswith(".jsonl"):     df = pd.read_json(path, lines=True)
    elif path.endswith(".json"):    df = pd.read_json(path)
    elif path.endswith(".parquet"): df = pd.read_parquet(path)
    elif path.endswith(".tsv"):     df = pd.read_csv(path, sep="\t")
    else:                           df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]
    df = df.astype(object).where(pd.notnull(df), None)
    return df.to_dict("records")

def make_submission(examples, path=OUT_PATH):
    start, unreadable = time.perf_counter(), 0
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["id", "label"])
        for i, ex in enumerate(examples):
            rid = ex.get("id")
            rid = f"{ex.get('dataset', 'row')}_{i}" if rid in (None, "") else rid
            if not ex["statement"] and not ex.get("conclusion_fol"):
                unreadable += 1; pred = False
            else:
                try:
                    pred = bool(predict(ex))
                except Exception:
                    pred = False                  # never crash on an unseen shape
            w.writerow([rid, pred])
    dt = time.perf_counter() - start
    print(f"wrote {path} | rows: {len(examples)} | unreadable: {unreadable} | "
          f"inference time: {dt:.2f}s | per sample: {dt / max(1, len(examples)):.6f}s")
    return unreadable

RULE_MODE = globals().get("RULE_MODE", "cwa")      # final setting (CWA unless changed above)

if os.path.exists(TEST_PATH):
    raw = load_test(TEST_PATH)
    print(len(raw), "rows | columns:", list(raw[0].keys()))
    examples = []
    for r in raw:
        e = normalize(r, r.get("dataset") or "test")
        if e["id"] is None:
            e["id"] = next((r[c] for c in ID_COLS if r.get(c) not in (None, "")), None)
        examples.append(e)
    bad = make_submission(examples)
    if bad: print("Some rows had no recognised context/statement column; add the names to CTX_KEYS / Q_KEYS.")
    sub = pd.read_csv(OUT_PATH)
    print(sub["label"].value_counts().to_string()); print("ids unique:", sub["id"].is_unique)
elif RUN_EXPERIMENTS:
    print(f"Test file '{TEST_PATH}' not found: writing predictions for all public samples instead.")
    make_submission([e for rows in DATA.values() for e in rows])
else:
    print(f"Test file '{TEST_PATH}' not found and public data not loaded: nothing written.")

Test file 'test.csv' not found: writing predictions for all public samples instead.
wrote submission.csv | rows: 4029 | unreadable: 0 | inference time: 139.69s | per sample: 0.034672s
